# Practice Lab: Multi-Provider API Integration (Lesson 7.1)

Module 7 · 8 exercises · LLM routing + cost optimization

Exercises 1, 4, 5, 6, 7, 8 run locally (pure Python).


# Lesson 7.1: Multi-Provider API Integration

8 exercises: 2E + 3M + 3C

Exercises 1 (SDK comparison), 4 (circuit breaker), 5 (batch costs), 6 (tiered router), 7 (caching), 8 (gateway) run **locally**.


In [ ]:
import os, time, json, re, math
from dataclasses import dataclass
from collections import defaultdict
from datetime import datetime
from enum import Enum


---
## Exercise 1 (Easy): Multi-Provider SDK Calls

Compare 3 providers: response structure, latency, tokens, cost.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from dataclasses import dataclass

@dataclass
class LLMResponse:
    text: str
    provider: str
    model: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    cost_usd: float

PRICING = {
    "gemini-2.0-flash":         {"input": 0.075, "output": 0.30},
    "gpt-4o":                   {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini":              {"input": 0.15,  "output": 0.60},
    "claude-sonnet-4-6": {"input": 3.00,  "output": 15.00},
    "claude-haiku-4-5":         {"input": 1.00,  "output": 5.00},
}

def calc_cost(model, in_tok, out_tok):
    p = PRICING.get(model, {"input": 0, "output": 0})
    return (in_tok * p["input"] + out_tok * p["output"]) / 1_000_000

responses = [
    LLMResponse("RAG retrieves relevant docs from a knowledge base, then feeds them as context to an LLM.",
                "gemini", "gemini-2.0-flash", 12, 28, 320, 0),
    LLMResponse("RAG combines information retrieval with text generation for accurate, sourced responses.",
                "openai", "gpt-4o", 14, 25, 890, 0),
    LLMResponse("Retrieval-Augmented Generation fetches external knowledge and feeds it to a language model.",
                "anthropic", "claude-sonnet-4-6", 13, 30, 1200, 0),
]

for r in responses:
    r.cost_usd = calc_cost(r.model, r.input_tokens, r.output_tokens)

print("Multi-Provider Comparison:")
print(f"{'Provider':<12} {'Model':<28} {'Latency':<10} "
      f"{'In Tok':<8} {'Out Tok':<9} {'Cost $':<10} "
      f"{'$/out_tok'}")
print("=" * 95)
for r in responses:
    per_tok = r.cost_usd / r.output_tokens if r.output_tokens else 0
    print(f"{r.provider:<12} {r.model:<28} {r.latency_ms:>6.0f}ms  "
          f"{r.input_tokens:<8} {r.output_tokens:<9} "
          f"${r.cost_usd:<9.6f} ${per_tok:.8f}")

cheapest = min(responses, key=lambda r: r.cost_usd)
fastest = min(responses, key=lambda r: r.latency_ms)
print(f"\nCheapest: {cheapest.provider} ({cheapest.model})")
print(f"Fastest:  {fastest.provider} ({fastest.model})")
print(f"\nGemini Flash is ~33-200x cheaper than GPT-4o/Claude Sonnet")


</details>


---
## Exercise 4 (Medium): Fallback + Circuit Breaker

3-state circuit breaker with automatic failover.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time
from enum import Enum

class CBState(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

class CircuitBreaker:
    def __init__(self, name, failure_threshold=3, cooldown_sec=5):
        self.name = name
        self.state = CBState.CLOSED
        self.failures = 0
        self.threshold = failure_threshold
        self.cooldown = cooldown_sec
        self.last_failure = 0
        self.successes_after_open = 0

    def can_execute(self):
        if self.state == CBState.CLOSED:
            return True
        if self.state == CBState.OPEN:
            if time.time() - self.last_failure >= self.cooldown:
                self.state = CBState.HALF_OPEN
                print(f"  [{self.name}] HALF_OPEN: testing")
                return True
            return False
        return self.state == CBState.HALF_OPEN

    def record_success(self):
        if self.state == CBState.HALF_OPEN:
            self.successes_after_open += 1
            if self.successes_after_open >= 2:
                self.state = CBState.CLOSED
                self.failures = 0
                self.successes_after_open = 0
                print(f"  [{self.name}] CLOSED: recovered!")
        self.failures = 0

    def record_failure(self):
        self.failures += 1
        self.last_failure = time.time()
        if self.failures >= self.threshold:
            self.state = CBState.OPEN
            print(f"  [{self.name}] OPEN: tripped after {self.failures} failures!")

    def status(self):
        return f"{self.name}: {self.state.value} (failures={self.failures})"

breakers = {
    "gpt-4o": CircuitBreaker("gpt-4o", failure_threshold=3, cooldown_sec=5),
    "claude-haiku": CircuitBreaker("claude-haiku"),
    "gemini-flash": CircuitBreaker("gemini-flash"),
}
chain = ["gpt-4o", "claude-haiku", "gemini-flash"]

def simulate_request(req_num, primary_fails=False):
    for model in chain:
        cb = breakers[model]
        if not cb.can_execute():
            print(f"  [{model}] SKIPPED (circuit open)")
            continue
        if model == "gpt-4o" and primary_fails:
            cb.record_failure()
            print(f"  [{model}] FAILED (failures={cb.failures})")
            continue
        cb.record_success()
        print(f"  [{model}] SUCCESS -> used")
        return model
    return "ALL_FAILED"

print("Fallback + Circuit Breaker Simulation:")
print("=" * 50)
for i in range(1, 11):
    primary_fails = i <= 5
    print(f"\nRequest {i}:")
    simulate_request(i, primary_fails)

print("\nCircuit Breaker Status:")
for name, cb in breakers.items():
    print(f"  {cb.status()}")


</details>


---
## Exercise 5 (Medium): Batch API Cost Savings

Calculate sync vs batch vs cache+batch costs.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

prompts = [{"id": f"req_{i:03d}", "est_input_tokens": 500,
            "est_output_tokens": 200} for i in range(100)]

total_in = sum(p["est_input_tokens"] for p in prompts)
total_out = sum(p["est_output_tokens"] for p in prompts)

print(f"Batch: {len(prompts)} prompts")
print(f"Total: {total_in:,} in + {total_out:,} out\n")

models = [
    ("GPT-4o", 2.50, 10.00),
    ("GPT-4o-mini", 0.15, 0.60),
    ("Claude Sonnet 4", 3.00, 15.00),
    ("Claude Haiku 3.5", 0.80, 4.00),
    ("Gemini Flash", 0.075, 0.30),
]

print(f"{'Model':<18} {'Sync $':<10} {'Batch $':<10} "
      f"{'Cache+Batch':<12} {'Savings'}")
print("=" * 65)
for name, inp, outp in models:
    sync = (total_in*inp + total_out*outp) / 1e6
    batch = sync * 0.50
    cache_rate = 0.10 if "Claude" in name else 0.50
    cached = ((total_in*inp*cache_rate + total_out*outp) / 1e6) * 0.50
    saving = (1 - cached/sync) * 100 if sync > 0 else 0
    print(f"{name:<18} ${sync:<9.4f} ${batch:<9.4f} "
          f"${cached:<11.4f} {saving:.0f}%")

print("\nMax savings: Claude Sonnet cache+batch = 95%")
print(f"$3.00/M -> $0.15/M input tokens")


</details>


---
## Exercise 6 (Challenge): Tiered Model Router

Complexity classification + cost-optimized routing.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

queries = [
    "What is the capital of France?",
    "What time is it in Mumbai?",
    "Translate 'hello' to Hindi",
    "How do I reset my password?",
    "What's the weather today?",
    "Summarize this document about AI ethics",
    "Compare React vs Vue for a project",
    "Write a SQL query for duplicate orders",
    "Explain REST vs GraphQL",
    "Help me draft an email to my manager",
    "Debug this Python function",
    "Create a marketing plan for a D2C brand",
    "Design a microservices architecture",
    "Analyze DPDPA pros and cons for startups",
    "Write a recursive tree traversal",
    "What are OWASP Top 10 for LLMs?",
    "Build a RAG pipeline with reranking",
    "Compare MHA vs GQA vs MLA attention",
    "Design a multi-tenant LLM gateway",
    "Explain quantum computing simply",
]

def classify(query):
    q = query.lower()
    complex_kw = ["design", "architecture", "build a",
                  "analyze", "recursive", "pipeline",
                  "multi-tenant", "mechanisms"]
    medium_kw = ["explain", "compare", "debug", "write",
                 "summarize", "create", "help me", "draft"]
    for kw in complex_kw:
        if re.search(kw, q):
            return "complex"
    for kw in medium_kw:
        if re.search(kw, q):
            return "medium"
    return "simple"

TIERS = {
    "simple":  {"model": "deepseek-chat", "input": 0.14, "output": 0.28},
    "medium":  {"model": "gpt-4o-mini", "input": 0.15, "output": 0.60},
    "complex": {"model": "claude-sonnet-4", "input": 3.00, "output": 15.00},
}

results = {"simple": 0, "medium": 0, "complex": 0}
print(f"{'#':<4} {'Tier':<10} {'Model':<18} {'Query':<48}")
print("-" * 82)
for i, q in enumerate(queries, 1):
    tier = classify(q)
    results[tier] += 1
    print(f"{i:<4} {tier:<10} {TIERS[tier]['model']:<18} {q[:46]}")

avg_in, avg_out = 200, 150
total = len(queries)

baseline = total * (avg_in*3.00 + avg_out*15.00) / 1e6
tiered = sum(
    count * (avg_in*TIERS[t]["input"] + avg_out*TIERS[t]["output"]) / 1e6
    for t, count in results.items()
)
savings = (1 - tiered/baseline) * 100

print(f"\nDistribution: {results}")
print(f"Baseline (all Sonnet): ${baseline:.6f}")
print(f"Tiered routing:        ${tiered:.6f} (-{savings:.0f}%)")

monthly_base = baseline * 10000/total * 30
monthly_tier = tiered * 10000/total * 30
print(f"\nMonthly (10K/day):")
print(f"  Baseline: Rs {monthly_base*84:,.0f}")
print(f"  Tiered:   Rs {monthly_tier*84:,.0f}")
print(f"  Savings:  Rs {(monthly_base-monthly_tier)*84:,.0f}/month")


</details>


---
## Exercise 7 (Challenge): Prompt Caching Implementation

Calculate caching economics across 3 providers.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
cached_tokens = 2000
user_tokens = 200
output_tokens = 150
num_calls = 1000

providers = [
    {"name": "Anthropic Claude Sonnet",
     "input": 3.00, "output": 15.00,
     "cache_write": 1.25, "cache_read": 0.10},
    {"name": "OpenAI GPT-4o",
     "input": 2.50, "output": 10.00,
     "cache_write": 1.00, "cache_read": 0.50},
    {"name": "Google Gemini Flash",
     "input": 0.075, "output": 0.30,
     "cache_write": 1.00, "cache_read": 0.10},
]

print("Prompt Caching Cost Analysis")
print(f"Cached prefix: {cached_tokens} tokens")
print(f"Fresh suffix: {user_tokens} tokens")
print(f"Output: {output_tokens} tokens")
print(f"Calls: {num_calls:,}\n")

print(f"{'Provider':<28} {'No Cache':<12} "
      f"{'With Cache':<12} {'Savings':<10} {'%'}")
print("-" * 65)

for p in providers:
    total_in = cached_tokens + user_tokens
    no_cache = num_calls * (
        total_in * p["input"] + output_tokens * p["output"]
    ) / 1e6

    write = 1 * cached_tokens * p["input"] * p["cache_write"] / 1e6
    reads = (num_calls-1) * cached_tokens * p["input"] * p["cache_read"] / 1e6
    fresh = num_calls * user_tokens * p["input"] / 1e6
    out_cost = num_calls * output_tokens * p["output"] / 1e6
    with_cache = write + reads + fresh + out_cost

    saved = no_cache - with_cache
    pct = saved / no_cache * 100

    print(f"{p['name']:<28} ${no_cache:<11.4f} "
          f"${with_cache:<11.4f} ${saved:<9.4f} {pct:.0f}%")

print("\nArchitecture: [CACHED PREFIX] + [DYNAMIC SUFFIX]")
print("  Cached: system prompt + examples + tools")
print("  Dynamic: conversation history + user message")


</details>


---
## Exercise 8 (Challenge): India-Compliant LLM Gateway

PII masking + DPDPA architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re, json

class PIIMasker:
    PATTERNS = {
        "aadhaar": (r"\b\d{4}\s?\d{4}\s?\d{4}\b",
                    "XXXX XXXX {last4}"),
        "pan": (r"\b[A-Z]{5}\d{4}[A-Z]\b",
                "XXXXX{mid4}X"),
        "phone": (r"\b(?:\+91|0)?[6-9]\d{9}\b",
                  "+91XXXXXX{last4}"),
        "email": (r"\b[\w.-]+@[\w.-]+\.\w+\b",
                  "{first2}***@***.***"),
    }

    @classmethod
    def mask(cls, text):
        masked = text
        detections = []
        for pii_type, (pattern, _) in cls.PATTERNS.items():
            for match in re.findall(pattern, masked):
                clean = match.replace(" ", "")
                if pii_type == "aadhaar":
                    repl = f"XXXX XXXX {clean[-4:]}"
                elif pii_type == "pan":
                    repl = f"XXXXX{clean[5:9]}X"
                elif pii_type == "phone":
                    repl = f"+91XXXXXX{clean[-4:]}"
                elif pii_type == "email":
                    repl = f"{clean[:2]}***@***.***"
                else:
                    repl = "***MASKED***"
                masked = masked.replace(match, repl)
                detections.append((pii_type, match, repl))
        return masked, detections

test = ("Customer Rajesh, Aadhaar 1234 5678 9012, "
        "PAN ABCDE1234F, phone +919876543210, "
        "email rajesh.kumar@gmail.com")
masked, dets = PIIMasker.mask(test)
print("PII Masking:")
print(f"  Original: {test}")
print(f"  Masked:   {masked}")
print(f"  Detections: {len(dets)}")
for t, o, r in dets:
    print(f"    {t}: {o} -> {r}")

gateway = {
    "layers": [
        {"name": "PII Middleware",
         "components": ["Aadhaar", "PAN", "Phone", "Email", "UPI"],
         "dpdpa": "Data minimization"},
        {"name": "Region Router",
         "components": ["Sensitive->Bedrock Mumbai",
                        "Financial->Azure Central India",
                        "Hindi->Sarvam AI", "General->Any"],
         "dpdpa": "Cross-border controls"},
        {"name": "Consent Manager",
         "components": ["Purpose-limited consent",
                        "Right to erasure", "Retention limits",
                        "Revocation API"],
         "dpdpa": "Consent requirements"},
        {"name": "Audit Logger",
         "components": ["Masked API call logs",
                        "Cost tracking", "DPIA",
                        "Monthly compliance reports"],
         "dpdpa": "Accountability"},
    ],
}

print("\nGateway Architecture:")
total_comp = 0
for i, layer in enumerate(gateway["layers"], 1):
    print(f"  Layer {i}: {layer['name']} [{layer['dpdpa']}]")
    for c in layer["components"]:
        print(f"    - {c}")
    total_comp += len(layer["components"])

print(f"\nLayers: {len(gateway['layers'])}")
print(f"Components: {total_comp}")

print("\nLatency: India region 5-20ms vs US 200-250ms")
print("Penalties: Rs 250 Cr | Deadline: May 2027")


</details>


---
## Exercise 2 (Easy): LiteLLM Unified Interface
Needs API keys / pip install. See practice-lab-7_1.html.


In [ ]:
# Exercise 2: LiteLLM Unified Interface
pass


---
## Exercise 3 (Medium): Structured Output Comparison
Needs API keys / pip install. See practice-lab-7_1.html.


In [ ]:
# Exercise 3: Structured Output Comparison
pass
